In [57]:
# Import libraries
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.layers.experimental import preprocessing

import numpy as np
import os
import time

In [58]:
# Load file data
text = ''

# TOLKIEN
text = open('./text_files/tolkien/tolkien.txt', 'rb').read().decode(encoding='utf-8')


print('Length of text: {} characters'.format(len(text)))

Length of text: 4012979 characters


In [59]:
# Verify the first part of our data
print(text[:250])


Concerning Hobbits 

This book is largely concerned with Hobbits, and from its pages a reader may discover much of their character and a little of their history. Further information will also be found in the selection from the Red Book of Westmar


In [60]:
# Now we'll get a list of the unique characters in the file. This will form the
# vocabulary of our ne200twork. There may be some characters we want to remove from this 
# set as we refine the network.
vocab = sorted(set(text))
print('{} unique characters'.format(len(vocab)))
print(vocab)

112 unique characters
['\t', '\n', '\r', ' ', '!', '"', '$', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', '~', '¨', '½', 'Á', 'É', 'Í', 'Ó', 'Ú', 'Û', 'á', 'â', 'ä', 'æ', 'é', 'ê', 'ë', 'í', 'î', 'ó', 'ô', 'ö', 'ú', 'û', 'ū', 'η', '†']


In [61]:
# Next, we'll encode encode these characters into numbers so we can use them
# with our neural network, then we'll create some mappings between the characters
# and their numeric representations
ids_from_chars = preprocessing.StringLookup(vocabulary=list(vocab))
chars_from_ids = tf.keras.layers.experimental.preprocessing.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True)

# Here's a little helper function that we can use to turn a sequence of ids
# back into a string:path_to_file
# turn them into a string:
def text_from_ids(ids):
  joinedTensor = tf.strings.reduce_join(chars_from_ids(ids), axis=-1)
  return joinedTensor.numpy().decode("utf-8")

In [62]:
# Now we'll verify that they work, by getting the code for "A", and then looking
# that up in reverse
testids = ids_from_chars(["T", "r", "u", "t", "h"])
testids

<tf.Tensor: shape=(5,), dtype=int64, numpy=array([51, 77, 80, 79, 67], dtype=int64)>

In [63]:
chars_from_ids(testids)

<tf.Tensor: shape=(5,), dtype=string, numpy=array([b'T', b'r', b'u', b't', b'h'], dtype=object)>

In [64]:
testString = text_from_ids( testids )
testString

'Truth'

In [65]:
# First, create a stream of encoded integers from our text
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
all_ids

<tf.Tensor: shape=(4012979,), dtype=int64, numpy=array([ 3,  2, 34, ..., 73, 66, 16], dtype=int64)>

In [66]:
# Now, convert that into a tensorflow dataset
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

In [67]:
# Finally, let's batch these sequences up into chunks for our training
seq_length = 100
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

# This function will generate our sequence pairs:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

# Call the function for every sequence in our list to create a new dataset
# of input->target pairs
dataset = sequences.map(split_input_target)

In [68]:
# Verify our sequences
for input_example, target_example in  dataset.take(1):
    print("Input: ", text_from_ids(input_example))
    print("--------")
    print("Target: ", text_from_ids(target_example))

Input:  
Concerning Hobbits 

This book is largely concerned with Hobbits, and from its pages a reader ma
--------
Target:  
Concerning Hobbits 

This book is largely concerned with Hobbits, and from its pages a reader may


In [69]:
# Finally, we'll randomize the sequences so that we don't just memorize the books
# in the order they were written, then build a new streaming dataset from that.
# Using a streaming dataset allows us to pass the data to our network bit by bit,
# rather than keeping it all in memory. We'll set it to figure out how much data
# to prefetch in the background.

BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

<PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

In [85]:
# Create our custom model. Given a sequence of characters, this
# model's job is to predict what character should come next.
class AustenTextModel(tf.keras.Model):

  # This is our class constructor method, it will be executed when
  # we first create an instance of the class 
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__(self)

    # Our model will have three layers:
    
    # 1. An embedding layer that handles the encoding of our vocabulary into
    #    a vector of values suitable for a neural network
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

    # 2. A GRU layer that handles the "memory" aspects of our RNN. If you're
    #    wondering why we use GRU instead of LSTM, and whether LSTM is better,
    #    take a look at this article: https://datascience.stackexchange.com/questions/14581/when-to-use-gru-over-lstm
    #    then consider trying out LSTM instead (or in addition to!)
    self.gru1 = tf.keras.layers.GRU(rnn_units, return_sequences=True, return_state=True)
    # self.dense2 = tf.keras.layers.Dense(vocab_size)
    self.gru2 = tf.keras.layers.GRU(rnn_units, return_sequences=True, return_state=True)

    # 3. Our output layer that will give us a set of probabilities for each
    #    character in our vocabulary.
    self.dense = tf.keras.layers.Dense(vocab_size)

  # This function will be executed for each epoch of our training. Here
  # we will manually feed information from one layer of our network to the 
  # next.
  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs

    # 1. Feed the inputs into the embedding layer, and tell it if we are
    #    training or predicting
    x = self.embedding(x, training=training)

    # 2. If we don't have any state in memory yet, get the initial random state
    #    from our GRUI layer.
    # x = self.dense2(x, training=training)
    if states is None:
      states1, states2 = (self.gru1.get_initial_state(x), self.gru2.get_initial_state(x))
    else:
      states1, states2 = states
    # 3. Now, feed the vectorized input along with the current state of memory
    #    into the gru layer.
    x, states1 = self.gru1(x, initial_state=states1, training=training)
    x, states2 = self.gru2(x, initial_state=states2, training=training)
    
    # 4. Finally, pass the results on to the dense layer
    x = self.dense(x, training=training)

    # 5. Return the results
    if return_state: return x, (states1, states2)
    else: return x

In [86]:
# Create an instance of our model
vocab_size=len(ids_from_chars.get_vocabulary())
embedding_dim = 256
rnn_units = 1024

model = AustenTextModel(vocab_size, embedding_dim, rnn_units)


In [87]:
# Verify the output of our model is correct by running one sample through
# This will also compile the model for us. This step will take a bit.
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

(64, 100, 113) # (batch_size, sequence_length, vocab_size)


In [88]:
# Now let's view the model summary
model.summary()

Model: "austen_text_model_10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_10 (Embedding)    multiple                  28928     
                                                                 
 gru_18 (GRU)                multiple                  3938304   
                                                                 
 gru_19 (GRU)                multiple                  6297600   
                                                                 
 dense_12 (Dense)            multiple                  115825    
                                                                 
Total params: 10,380,657
Trainable params: 10,380,657
Non-trainable params: 0
_________________________________________________________________


In [89]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss)

In [90]:
checkpoint = keras.callbacks.ModelCheckpoint("./checkpoint",monitor='loss', save_best_only=True, mode='min', verbose=True)
early_stop = keras.callbacks.EarlyStopping(monitor='loss', patience=50)

In [92]:
history = model.fit(dataset, epochs=10, callbacks=[early_stop, checkpoint])

Epoch 1/10
620/620 [==============================] - ETA: 0s - loss: 1.7315
Epoch 1: loss improved from inf to 1.73150, saving model to .\checkpoint


INFO:tensorflow:Assets written to: .\checkpoint\assets


INFO:tensorflow:Assets written to: .\checkpoint\assets


620/620 [==============================] - 262s 421ms/step - loss: 1.7315
Epoch 2/10
 96/620 [===>..........................] - ETA: 3:35 - loss: 1.3151

In [ ]:
# ONLY SAVE WEIGHTS IF YOU WANT TO OVERRIDE THE PREVIOUS MODEL
# model.save_weights(weight_dir)

In [ ]:
# Here's the code we'll use to sample for us. It has some extra steps to apply
# the temperature to the distribution, and to make sure we don't get empty
# characters in our text. Most importantly, it will keep track of our model
# state for us.

class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature=temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Create a mask to prevent "" or "[UNK]" from being generated.
    skip_ids = self.ids_from_chars(['','[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        # Put a -inf at each bad index.
        values=[-float('inf')]*len(skip_ids),
        indices = skip_ids,
        # Match the shape to the vocabulary
        dense_shape=[len(ids_from_chars.get_vocabulary())]) 
    self.prediction_mask = tf.sparse.to_dense(sparse_mask,validate_indices=False)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Convert strings to token IDs.
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Run the model.
    # predicted_logits.shape is [batch, char, next_char_logits] 
    predicted_logits, states =  self.model(inputs=input_ids, states=states, 
                                          return_state=True)
    # Only use the last prediction.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    
    # Apply the prediction mask: prevent "" or "[UNK]" from being generated.
    predicted_logits = predicted_logits + self.prediction_mask

    # Sample the output logits to generate token IDs.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Return the characters and model state.
    return chars_from_ids(predicted_ids), states


In [ ]:
# If you're loading in a model
# loaded_model = AustenTextModel(vocab_size, embedding_dim, rnn_units)
# for input_example_batch, target_example_batch in dataset.take(1):
#     example_batch_predictions = loaded_model(input_example_batch)
#     print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")
# loaded_model.load_weights(weight_dir)

(64, 100, 90) # (batch_size, sequence_length, vocab_size)


In [ ]:
# Create an instance of the character generator

# IF NOT LOADED IN
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

# IF LOADED IN
# one_step_model = OneStep(loaded_model, chars_from_ids, ids_from_chars)

# Now, let's generate a 1000 character chapter by giving our model "Chapter 1"
# as its starting text
states = None
next_char = tf.constant(['Chapter 1'])
result = [next_char]

for n in range(1000):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)


result = tf.strings.join(result)

# Print the results formatted.
print(result[0].numpy().decode('utf-8'))

Chapter 1 

THE FIRER 

IN the echoing pass I goes, where Steads has gone by him, they should have found some other erch: been corrected by crooke-pointing almost. I don't like the Ring. But now without voice was not him!' 

'Who are the move, fourty---if you can find it, but I do not tell us ic?' he said. 'As for I can sleep, it looks easi) to death; but black that live derived from the southern face of the hills. There the River Gollum was hope,' said Frodo, leaving little heering behind, they saw nothing, opening it slowly, but with another pace was sinking as a word of silver and more along with small swift humbled axe: wholly arrows still deeds, falling in.' 

'Now, Sam, Gollum!' he said. 'Tell us enough. 

Some of the trees, a warious regretting from Mirkwood. Did you skin, or joy. We will follow me from the bushes, we asked how the night was just rememed by the men of the Rings. The Sun was bent, unmerramations of plainly the brown slime of Círdan were hastened in the air, but 
